### Transform Payments Data
- Extract Date and Time from the payment_timestamp and create new columns payment_date and payement_time
- Map payment_status to contain descriptive values (1-Success, 2-Pending, 3- Cancelled, 4- Failed)
- Write transformed data into silver layer

#### 1. Extract Date and Time from the payment_timestamp and create new columns payment_date and payement_time
https://docs.databricks.com/aws/en/sql/language-manual/functions/date_format


In [0]:
%sql
SELECT
    *,
    DATE(payment_timstamp) AS payement_date,
    date_format(payment_timstamp, 'HH:mm:ss') AS payement_time
FROM gizmobox.bronze.payments
ORDER BY payment_id;

In [0]:
%sql
SELECT
    *,
    CAST(date_format(payment_timstamp, 'yyyy-MM-dd') AS DATE) AS payement_date,
    date_format(payment_timstamp, 'HH:mm:ss') AS payement_time
FROM gizmobox.bronze.payments
ORDER BY payment_id;

#### 2. Map payment_status to contain descriptive values (1-Success, 2-Pending, 3- Cancelled, 4- Failed)


In [0]:
%sql
SELECT
    payment_id,
    order_id,
    payment_timstamp,
    payement_status,
    payment_method,
    DATE(payment_timstamp) AS payement_date,
    date_format(payment_timstamp, 'HH:mm:ss') AS payement_time,
    CASE 
        WHEN payement_status = 1 THEN 'SUCCESS'
        WHEN payement_status = 2 THEN 'PENDING'
        WHEN payement_status = 3 THEN 'CANCELLED'
        WHEN payement_status = 4 THEN 'FAILED'
        ELSE 'UNKNOWN'
    END AS payement_status_str
FROM gizmobox.bronze.payments
ORDER BY payment_id;

In [0]:
%sql
SELECT
    payment_id,
    order_id,
    payment_timstamp,
    payement_status,
    payment_method,
    CAST(date_format(payment_timstamp, 'yyyy-MM-dd') AS DATE) AS payement_date,
    date_format(payment_timstamp, 'HH:mm:ss') AS payement_time,
    CASE payement_status
        WHEN  1 THEN 'SUCCESS'
        WHEN  2 THEN 'PENDING'
        WHEN  3 THEN 'CANCELLED'
        WHEN  4 THEN 'FAILED'
        ELSE 'UNKNOWN'
    END AS payement_status_str
FROM gizmobox.bronze.payments
ORDER BY payment_id;

#### 3. Write transformed data into silver layer

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.silver.payments
AS
SELECT
    payment_id,
    order_id,
    CAST(date_format(payment_timstamp, 'yyyy-MM-dd') AS DATE) AS payement_date,
    date_format(payment_timstamp, 'HH:mm:ss') AS payement_time,
    CASE payement_status
        WHEN  1 THEN 'SUCCESS'
        WHEN  2 THEN 'PENDING'
        WHEN  3 THEN 'CANCELLED'
        WHEN  4 THEN 'FAILED'
        ELSE 'UNKNOWN'
    END AS payement_status_str,
    payment_method
FROM gizmobox.bronze.payments
ORDER BY payment_id;

In [0]:
%sql
SELECT * FROM gizmobox.silver.payments;